# Device hardening  (MODEL SOLUTION, instructor copy, do not distribute)

# Coding problem, 4.3.a: Device hardening configuration auditor

**Skill:** Mitigate Risk · **Maps to:** 4.3.A · **Language:** Python 3 (pytest)

## Problem
In a secure enterprise environment, devices must be configured according to strict security baselines before deployment. Over time, administrators or software installers may change settings, resulting in "configuration drift" that increases the device's attack surface.

Your task is to write a python function `solve(data)` that takes a target device's actual configuration and audit it against a security baseline. The function must return a list of all check names that the configuration fails, sorted alphabetically.

## Function contract
Implement `solve(data)` in the code editor (see the Reference section below).

## Input
A dictionary with keys `"baseline"` and `"configuration"`.
Example:
```json
{
  "baseline": {
    "min_password_length": 12,
    "disallowed_services": ["ftp", "telnet"],
    "firewall_enabled": true,
    "encryption_enabled": true,
    "allowed_accounts": ["admin", "operator"]
  },
  "configuration": {
    "password_length": 8,
    "services": ["ssh", "ftp"],
    "firewall_enabled": false,
    "encryption_enabled": true,
    "accounts": ["admin", "guest", "operator"]
  }
}
```

## Output
A list of strings representing the names of the failing checks from the baseline, sorted alphabetically.
Example output for the input above:
```json
["allowed_accounts", "disallowed_services", "firewall_enabled", "min_password_length"]
```
*(Explanation: password length 8 is less than 12; ftp service is disallowed; firewall is false; guest account is not allowed)*

## Constraints
- If an expected configuration field is missing, default values apply:
  - `password_length`: `0`
  - `services`: `[]`
  - `firewall_enabled`: `False`
  - `encryption_enabled`: `False`
  - `accounts`: `[]`
- Standard library only.
- Case-sensitive string matching for services and accounts.

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(data: dict) -> list[str]
```

### Input (`data`)
A dictionary with two keys: `"baseline"` and `"configuration"`.
The `"baseline"` dictionary contains the security requirements:
- `"min_password_length"` (int)
- `"disallowed_services"` (list of strings)
- `"firewall_enabled"` (bool)
- `"encryption_enabled"` (bool)
- `"allowed_accounts"` (list of strings)

The `"configuration"` dictionary contains the target device's actual settings:
- `"password_length"` (int, optional - default is `0` if missing)
- `"services"` (list of strings, optional - default is empty if missing)
- `"firewall_enabled"` (bool, optional - default is `False` if missing)
- `"encryption_enabled"` (bool, optional - default is `False` if missing)
- `"accounts"` (list of strings, optional - default is empty if missing)

### Output (`result`)
A list of strings representing the names of the checks from the baseline that failed.
The possible check names in the output list are:
- `"min_password_length"` (fails if configuration password length is less than baseline minimum)
- `"disallowed_services"` (fails if any service in the configuration is present in the baseline disallowed list)
- `"firewall_enabled"` (fails if baseline requires it but configuration has it set to False)
- `"encryption_enabled"` (fails if baseline requires it but configuration has it set to False)
- `"allowed_accounts"` (fails if configuration contains any accounts NOT present in the baseline allowed list)

The output list must be sorted alphabetically. If all checks pass, returns an empty list `[]`.

### Guarantees / rules
- Pure function, deterministic, no I/O, no network.
- Standard library only.
- Case-sensitivity: comparisons for services and accounts are case-sensitive.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Reference solution
def solve(data: dict) -> list[str]:
    """Audit a device's configuration against a security baseline.

    Args:
        data: A dict with keys "baseline" and "configuration".

    Returns:
        A list of failing baseline check names, sorted alphabetically.
    """
    baseline = data.get("baseline", {})
    config = data.get("configuration", {})

    failed = []

    # Check 1: min_password_length
    min_pw = baseline.get("min_password_length", 0)
    actual_pw = config.get("password_length", 0)
    if actual_pw < min_pw:
        failed.append("min_password_length")

    # Check 2: disallowed_services
    disallowed_srv = set(baseline.get("disallowed_services", []))
    actual_srv = config.get("services", [])
    if any(s in disallowed_srv for s in actual_srv):
        failed.append("disallowed_services")

    # Check 3: firewall_enabled
    fw_required = baseline.get("firewall_enabled", False)
    fw_actual = config.get("firewall_enabled", False)
    if fw_required and not fw_actual:
        failed.append("firewall_enabled")

    # Check 4: encryption_enabled
    enc_required = baseline.get("encryption_enabled", False)
    enc_actual = config.get("encryption_enabled", False)
    if enc_required and not enc_actual:
        failed.append("encryption_enabled")

    # Check 5: allowed_accounts
    allowed_acc = set(baseline.get("allowed_accounts", []))
    actual_acc = config.get("accounts", [])
    if any(a not in allowed_acc for a in actual_acc):
        failed.append("allowed_accounts")

    return sorted(failed)


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

import pytest


# ---- visible cases -------------------------------------------------------
def test_example_1():
    data = {
        "baseline": {
            "min_password_length": 12,
            "disallowed_services": ["ftp", "telnet"],
            "firewall_enabled": True,
            "encryption_enabled": True,
            "allowed_accounts": ["admin", "operator"]
        },
        "configuration": {
            "password_length": 8,
            "services": ["ssh", "ftp"],
            "firewall_enabled": False,
            "encryption_enabled": True,
            "accounts": ["admin", "guest", "operator"]
        }
    }
    expected = ["allowed_accounts", "disallowed_services", "firewall_enabled", "min_password_length"]
    assert impl.solve(data) == expected


def test_example_2():
    data = {
        "baseline": {
            "min_password_length": 8,
            "disallowed_services": ["telnet"],
            "firewall_enabled": True,
            "encryption_enabled": False,
            "allowed_accounts": ["admin"]
        },
        "configuration": {
            "password_length": 10,
            "services": ["ssh", "http"],
            "firewall_enabled": True,
            "encryption_enabled": True,
            "accounts": ["admin"]
        }
    }
    expected = []
    assert impl.solve(data) == expected


# ---- hidden / edge cases -------------------------------------------------
def test_edge_empty():
    # Empty configurations default to fail checks that are required
    data = {
        "baseline": {
            "min_password_length": 10,
            "disallowed_services": ["ftp"],
            "firewall_enabled": True,
            "encryption_enabled": True,
            "allowed_accounts": ["admin"]
        },
        "configuration": {}
    }
    # password_length defaults to 0 (fails)
    # services defaults to [] (passes - no disallowed service present)
    # firewall_enabled defaults to False (fails)
    # encryption_enabled defaults to False (fails)
    # accounts defaults to [] (passes - no unauthorized accounts present)
    expected = ["encryption_enabled", "firewall_enabled", "min_password_length"]
    assert impl.solve(data) == expected


def test_all_pass():
    data = {
        "baseline": {
            "min_password_length": 14,
            "disallowed_services": ["ftp", "telnet", "http", "smb"],
            "firewall_enabled": True,
            "encryption_enabled": True,
            "allowed_accounts": ["root", "deploy"]
        },
        "configuration": {
            "password_length": 14,
            "services": ["ssh", "https"],
            "firewall_enabled": True,
            "encryption_enabled": True,
            "accounts": ["root"]
        }
    }
    assert impl.solve(data) == []


def test_case_sensitivity():
    data = {
        "baseline": {
            "min_password_length": 8,
            "disallowed_services": ["Telnet"],
            "firewall_enabled": False,
            "encryption_enabled": False,
            "allowed_accounts": ["Admin"]
        },
        "configuration": {
            "password_length": 8,
            "services": ["telnet"], # disallowed has 'Telnet', case-sensitive means passes
            "accounts": ["admin"] # allowed has 'Admin', case-sensitive means fails
        }
    }
    assert impl.solve(data) == ["allowed_accounts"]


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
